In [1]:
import pandas as pd
from matplotlib import pyplot as plt
from prophet import Prophet
from datetime import datetime
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
import matplotlib.pyplot as plt
import itertools

/Users/jisu/anaconda3/envs/practicum/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# pl.Config(fmt_str_lengths=1000, tbl_width_chars=1000)
pd.set_option('display.max_columns', None)
df = pd.read_csv('./data/ecommerce_data.csv')

# Clean the data (drop null)
df = df.dropna()

df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])

df = df.rename(columns={"Transaction_Date": "ds"})

print(df.head())


# North America Data
df_na = df[(df['Region']=='North America')]
print(df_na)

                         Transaction_ID     Customer_ID   Product_ID  \
0  8b460852-7c64-46fa-998b-b0976879d082     Customer_65  Product_224   
1  418612e7-8744-4ba3-bb0c-105b47e2a968   Customer_1910  Product_584   
2  5bc3b98f-cb0c-4b12-947c-df8bbb35a73e   Customer_2306  Product_374   
3  28fb67c8-e8c0-447a-841c-f760730de0eb  Customer_17206  Product_220   
4  8bee087a-a8a9-45bb-89d7-04d1710f1b00  Customer_16033  Product_358   

          ds  Units_Sold  Discount_Applied  Revenue  Clicks  Impressions  \
0 2024-10-06         134              0.14   305.54      11           65   
1 2024-10-29         109              0.30  1102.19      15          201   
2 2024-04-04         116              0.04   471.29      16          199   
3 2024-08-25         125              0.20   980.26      12          355   
4 2024-05-05         132              0.07   803.76      44          355   

   Conversion_Rate         Category         Region  Ad_CTR  Ad_CPC  Ad_Spend  
0             0.17      Electro

In [3]:
print(df['ds'].value_counts().head(10))

ds
2024-10-14    319
2024-09-13    313
2024-10-19    312
2024-04-11    310
2024-01-21    309
2024-05-28    306
2024-01-03    305
2024-07-15    305
2024-01-19    304
2024-06-17    304
Name: count, dtype: int64


In [4]:
m = Prophet(interval_width = 0.90)
m.add_country_holidays(country_name='US')

m.add_regressor('Ad_Spend')
m.add_regressor('Clicks')
m.add_regressor('Impressions')

m.fit(df_daily)


future = m.make_future_dataframe(periods=30)
future['Ad_Spend'] = df_daily['Ad_Spend'].mean()
future['Clicks'] = df_daily['Clicks'].mean()
future['Impressions'] = df_daily['Impressions'].mean()


forecast = m.predict(future)
print(forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail())
# print(forecast.tail())

fig = m.plot(forecast)
plt.show()

NameError: name 'df_daily' is not defined

In [ ]:
df_cv = cross_validation(
    m, 
    initial='180 days',   
    period='60 days',    
    horizon='30 days'    
)

# Calculate performance metrics
df_p = performance_metrics(df_cv)

In [ ]:
print(df_cv)
# print(df_p)

In [ ]:
print(df_p)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(df_p['horizon'].astype(str), df_p['mape'], marker='o')
plt.xticks(rotation=45)
plt.ylabel('MAPE')
plt.title('MAPE vs Forecast Horizon')
plt.show()

In [ ]:
fig2 = m.plot_components(forecast)
plt.show()

In [ ]:
# Including Black Friday
black_friday = pd.DataFrame({
  'holiday': 'black_friday',
  'ds': pd.to_datetime(['2024-11-29']),
  'lower_window': 0,
  'upper_window': 3, 
})

m1 = Prophet(interval_width = 0.90, holidays=black_friday)

m1.add_country_holidays(country_name='US')

m1.add_regressor('Ad_Spend')
m1.add_regressor('Clicks')
m1.add_regressor('Impressions')

m1.fit(df_daily)


future = m1.make_future_dataframe(periods=30)
future['Ad_Spend'] = df_daily['Ad_Spend'].mean()
future['Clicks'] = df_daily['Clicks'].mean()
future['Impressions'] = df_daily['Impressions'].mean()


forecast = m1.predict(future)
print(forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail())
# print(forecast.tail())

fig = m1.plot(forecast)
plt.show()

In [ ]:
df_cv_2 = cross_validation(
    m1, 
    initial='60 days',   
    period='30 days',    
    horizon='30 days'    
)

# Calculate performance metrics
df_p_2 = performance_metrics(df_cv)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(df_p_2['horizon'].astype(str), df_p_2['mape'], marker='o')
plt.xticks(rotation=45)
plt.ylabel('MAPE')
plt.title('MAPE vs Forecast Horizon Including Black Friday')
plt.show()

In [ ]:
fig2 = m1.plot_components(forecast)
plt.show()

# North America Data

In [ ]:
print(df.columns)

In [ ]:
# North America Data
df = df[(df['Region']=='North America')]

# df_daily = df.groupby('ds')['Revenue'].sum().reset_index()
df_daily = df.groupby('ds').agg({
    'Revenue': 'sum',
    'Ad_Spend': 'sum',
    'Clicks': 'sum',
    'Impressions': 'sum',
    'Discount_Applied': 'sum',
    'Conversion_Rate': 'sum',
    'Ad_CTR': 'sum',
    'Ad_CPC': 'sum',
    'Ad_Spend': 'sum'
}).reset_index()


df_daily = df_daily.sort_values('ds')
df_daily = df_daily.rename(columns={'Revenue': 'y'})

# print(df_daily)
print(df_daily.columns)

# Including Black Friday
black_friday = pd.DataFrame({
  'holiday': 'black_friday',
  'ds': pd.to_datetime(['2024-11-29']),
  'lower_window': 0,
  'upper_window': 3, 
})

m = Prophet(interval_width = 0.90, holidays=black_friday)

m.add_country_holidays(country_name='US')

regressors = ['Ad_Spend', 'Clicks', 'Impressions', 'Discount_Applied', 'Conversion_Rate', 'Ad_CTR', 'Ad_CPC', 'Ad_Spend']

for regressor in regressors:
    m.add_regressor(regressor)

m.fit(df_daily)

future = m.make_future_dataframe(periods=30)

for futures in regressors:
    future[futures] = df_daily[futures].mean()
    
# future['Ad_Spend'] = df_daily['Ad_Spend'].mean()
# future['Clicks'] = df_daily['Clicks'].mean()
# future['Impressions'] = df_daily['Impressions'].mean()


forecast = m.predict(future)
print(forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail())
# print(forecast.tail())

fig = m.plot(forecast)
plt.show()



df_cv = cross_validation(
    m, 
    initial='90 days',   
    period='30 days',    
    horizon='30 days'    
)

# Calculate performance metrics
df_p_2 = performance_metrics(df_cv)
print(df_p_2)

plt.figure(figsize=(8,5))
plt.plot(df_p_2['horizon'].astype(str), df_p_2['mape'], marker='o')
plt.xticks(rotation=45)
plt.ylabel('MAPE')
plt.title('MAPE vs Forecast Horizon Including Black Friday (NA)')
plt.show()

fig2 = m.plot_components(forecast)
plt.show()

In [ ]:
print(df_cv)
m.plot(df_cv)

In [ ]:
# df_daily = df.groupby('ds')['Revenue'].sum().reset_index()
df_daily = df.groupby('ds').agg({
    'Revenue': 'sum',
    'Ad_Spend': 'sum',
    'Clicks': 'sum',
    'Impressions': 'sum',
    'Conversion_Rate': 'sum',
    'Ad_CTR': 'sum',
    'Ad_CPC': 'sum'
}).reset_index()


df_daily = df_daily.sort_values('ds')
df_daily = df_daily.rename(columns={'Revenue': 'y'})

print(df_daily)
print(df_daily.columns)

In [ ]:
df_daily.columns

In [ ]:
# Including Black Friday
black_friday = pd.DataFrame({
  'holiday': 'black_friday',
  'ds': pd.to_datetime(['2024-11-29']),
  'lower_window': 0,
  'upper_window': 3, 
})


In [ ]:

# Model 2 (With Regressors)

m = Prophet(interval_width = 0.90, 
            holidays=black_friday,
           seasonality_mode='multiplicative')

m.add_seasonality(name='weekly', period=7, fourier_order=4, prior_scale=0.01)
m.add_country_holidays(country_name='US')

regressors = ['Clicks', 'Impressions', 'Conversion_Rate', 'Ad_CTR', 'Ad_CPC', 'Ad_Spend']

for regressor in regressors:
    m.add_regressor(regressor)

m.fit(df_daily)

future = m.make_future_dataframe(periods=30)

for futures in regressors:
    future[futures] = df_daily[futures].mean()

forecast = m.predict(future)
# print(forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail())
# print(forecast.tail())

# fig = m.plot(forecast)
# plt.show()

df_cv = cross_validation(
    m, 
    initial='270 days',   
    period='60 days',    
    horizon='30 days'    
)

# Calculate performance metrics
df_p = performance_metrics(df_cv)
print(df_p)

fig2 = m.plot_components(forecast)
plt.show()


# Calculate Accuracy

overall_accuracy = df_p['mape'].mean() *100

print(f"Accuracy (30 days): {overall_accuracy:.2f}%")


In [ ]:
# Model 3 (Hyperparameter Test)
# REFERENCE: https://velog.io/@sooyeon/prophet-crossvalidation

hyperparameters = {  
    'changepoint_prior_scale': [0.001, 0.01, 0.1, 0.5],
    'seasonality_prior_scale': [0.01, 0.1, 1.0, 10.0],
    'holidays_prior_scale': [0.01, 0.1, 1.0]
}

all_hyperparam = [dict(zip(hyperparameters.keys(), v)) for v in itertools.product(*hyperparameters.values())]
mape = []


for hyper in all_hyperparam:
    m = Prophet(**hyper,
                interval_width = 0.90, 
                holidays=black_friday,
               seasonality_mode='multiplicative')
    
    m.add_seasonality(name='weekly', period=7, fourier_order=4, prior_scale=0.01)
    m.add_country_holidays(country_name='US')
    
    regressors = ['Clicks', 'Impressions', 'Conversion_Rate', 'Ad_CTR', 'Ad_CPC', 'Ad_Spend']
    
    for regressor in regressors:
        m.add_regressor(regressor)
    
    m.fit(df_daily)
    
    future = m.make_future_dataframe(periods=30)
    
    for futures in regressors:
        future[futures] = df_daily[futures].mean()
    
    forecast = m.predict(future)
    # print(forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail())
    # print(forecast.tail())
    
    # fig = m.plot(forecast)
    # plt.show()
    
    df_cv = cross_validation(
        m, 
        initial='270 days',   
        period='60 days',    
        horizon='30 days'    
    )
    
    # Calculate performance metrics
    df_p = performance_metrics(df_cv)
    # print(df_p)
    
    # fig2 = m.plot_components(forecast)
    # plt.show()
    
    
    # Calculate Accuracy
    # df_p['accuracy_smape'] = (1 - (df_p['smape'] / 2)) * 100
    
    overall_accuracy = df_p['mape'].mean() *100
    mape.append(overall_accuracy)

    # print(f"Accuracy (30 days): {overall_accuracy:.2f}%")


results = pd.DataFrame(all_hyperparam)
results['accuracy'] = mape
print(results)


In [ ]:
results.sort_values(by='accuracy', ascending=True)

In [ ]:

m2 = Prophet(changepoint_prior_scale=0.001,
             seasonality_prior_scale=0.01,
             holidays_prior_scale=1,
            interval_width = 0.90, 
            holidays=black_friday,
           seasonality_mode='multiplicative')

m2.add_seasonality(name='weekly', period=7, fourier_order=4, prior_scale=0.01)
m2.add_country_holidays(country_name='US')

regressors = ['Clicks', 'Impressions', 'Conversion_Rate', 'Ad_CTR', 'Ad_CPC', 'Ad_Spend']

for regressor in regressors:
    m2.add_regressor(regressor)

m2.fit(df_daily)

future = m2.make_future_dataframe(periods=30)

for futures in regressors:
    future[futures] = df_daily[futures].mean()

forecast = m2.predict(future)
# print(forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail())
print(forecast.tail())

# fig = m2.plot(forecast)
# plt.show()

df_cv_2 = cross_validation(
    m2, 
    initial='270 days',   
    period='60 days',    
    horizon='30 days'    
)

# Calculate performance metrics
df_p_2 = performance_metrics(df_cv_2)
print(df_p_2)

fig2 = m2.plot_components(forecast)
plt.show()


# Calculate Accuracy

overall_accuracy = df_p_2['mape'].mean() *100

print(f"Accuracy (30 days): {overall_accuracy:.2f}%")

In [ ]:
fig = m2.plot(forecast)
fig.suptitle("Revenue Forecast (Prophet Model)", fontsize=16, y=1.02)
plt.show()


In [ ]:
overall_mape = df_p_2['mape'].mean()
overall_rmse = df_p_2['rmse'].mean()
overall_mae = df_p_2['mae'].mean()


print(f"Average MAPE: {overall_mape:.4f}")
print(f"Average RMSE: {overall_rmse:.2f}")
print(f"Average MAE: {overall_mae:.2f}")


print(f"Accuracy (MAPE): {(1 - overall_mape) * 100:.2f}%")

In [ ]:
fig2 = m2.plot_components(forecast)
fig2.suptitle("Trend of the Prophet Model", fontsize=16, y=1.02)
plt.show()

In [ ]:
m2.plot(df_cv_2)